# Green sight · Green 360° ML Pipeline

### Arquitectura del Sistema de Clasificación (Clustering)

Decidimos implementar un sistema basado en **Clustering** conectado directamente a los datos reales de **Green Services (Green 360°)** para realizar las siguientes clasificaciones estratégicas:

1. **Predicción y segmentación de clientes** (PCA + K-Means + GLM Tweedie + MoE).
2. **Predicción de productos de mayor necesidad / demanda** (portafolio documental, respaldo, licencias, continuidad y soporte de Green Services).

#### Componentes Tecnológicos:
* **K-Means & PCA:** Para el análisis, reducción de dimensionalidad y agrupación de clientes y servicios.
* **GLM Tweedie (`var_power=1.5`):** Para modelar demanda sesgada a la derecha con ceros reales en clientes inactivos.
* **MoE (Mixture of Experts) en PyTorch:** Orquestación neuronal con *Gating Network* especializada por clúster (`Alto Valor`, `Recurrente`, `Reactivación`).
* **Hugging Face + Docker con n8n + MCP:** Generación de prompts comerciales contextualizados por el MoE (restringidos exclusivamente a Green Services) y automatización de campañas de correo masivo vía webhook n8n.

### Hoja de Ruta del Modelado: Pipeline de Machine Learning para MCP

```
[Datos Reales Green 360° (clientes, ventas, TRM)]
       │
       ▼
[PCA (Reducción de Dimensionalidad · 3 Componentes)]
       │
       ▼
[K-Means Clustering (k=3)] ───> (Alto Valor / Recurrente / En Riesgo)
       │
       ▼
[GLM (Modelos Lineales con Gamma/Tweedie p=1.5)] ───> (Demanda Base por Clúster y Producto)
       │
       ▼
[Mixture of Experts (MoE)] ───> (Orquestación Óptima + Pesos de Expertos)
       │                              ├──> [Hugging Face LLM] (Prompt MoE solo Green Services)
       │                              └──> [Docker + n8n] (Automatización de Correo Masivo)
       ▼
[Servidor MCP + API FastAPI] ───> (Dashboard Green 360° & Agente Operativo)
```

In [ ]:
# 1. Dependencias (ejecutar en Google Colab si hace falta)
# !pip install -q scikit-learn statsmodels torch transformers accelerate mcp fastapi uvicorn requests pandas numpy matplotlib seaborn pyngrok

import json
import os
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
print("Entorno GreenSight ML listo.")

In [ ]:
# 2. Carga de datos reales de Green 360° (local del repo o directo desde GitHub)
REPO_RAW_BASE = "https://raw.githubusercontent.com/Mrcsbda/green-360/main/src/data/fixtures"
LOCAL_FIXTURES = Path("../src/data/fixtures")
FECHA_REFERENCIA = "2026-10-31"
REF_DT = pd.to_datetime(FECHA_REFERENCIA)

def load_json_resource(filename: str):
    local_path = LOCAL_FIXTURES / filename
    if local_path.exists():
        return json.loads(local_path.read_text(encoding="utf-8"))
    url = f"{REPO_RAW_BASE}/{filename}"
    resp = requests.get(url, timeout=20)
    resp.raise_for_status()
    return resp.json()

customers_raw = load_json_resource("customers.json")
sales_raw = load_json_resource("sales.json")
trm_raw = load_json_resource("trm.json")

df_customers = pd.DataFrame(customers_raw)
df_sales = pd.DataFrame(sales_raw)
df_trm = pd.DataFrame(trm_raw, columns=["fecha", "cop_por_usd"]).sort_values("fecha")
df_trm["fecha_dt"] = pd.to_datetime(df_trm["fecha"])

# Conversión USD -> COP con la TRM diaria exacta o más reciente anterior
df_sales["fecha_dt"] = pd.to_datetime(df_sales["fecha"])
df_sales = df_sales.sort_values("fecha_dt")
df_sales = pd.merge_asof(
    df_sales,
    df_trm[["fecha_dt", "cop_por_usd"]],
    on="fecha_dt",
    direction="backward",
)
df_sales["cop_por_usd"] = df_sales["cop_por_usd"].fillna(df_trm["cop_por_usd"].iloc[0])
df_sales["valor_cop"] = np.where(
    df_sales["moneda"] == "USD",
    df_sales["cantidad"] * df_sales["precioUnitario"] * df_sales["cop_por_usd"],
    df_sales["cantidad"] * df_sales["precioUnitario"],
)

df_conf = df_sales[
    (df_sales["estado"] == "Confirmada") & (df_sales["fecha_dt"] <= REF_DT)
].copy()

print(f"Clientes cargados: {len(df_customers)}")
print(f"Ventas confirmadas al {FECHA_REFERENCIA}: {len(df_conf)}")
print(f"Facturación histórica confirmada: ${df_conf['valor_cop'].sum() / 1e6:,.2f} M COP")

In [ ]:
# 3. Ingeniería de características: Clientes y Productos Green Services
rows_cust = []
for _, c in df_customers.iterrows():
    mine = df_conf[df_conf["customerId"] == c["id"]].sort_values("fecha_dt")
    days_ago = (REF_DT - mine["fecha_dt"]).dt.days if len(mine) else pd.Series([], dtype=float)
    in12 = mine[days_ago < 365] if len(mine) else mine
    in90 = mine[days_ago < 91] if len(mine) else mine
    prev_q = mine[(days_ago >= 91) & (days_ago < 456)] if len(mine) else mine

    freq_12m = float(len(in12))
    ticket_m = float(mine["valor_cop"].mean() / 1e6) if len(mine) else 0.0
    recencia = float((REF_DT - mine["fecha_dt"].max()).days) if len(mine) else 365.0
    variedad = float(mine["producto"].nunique()) if len(mine) else 0.0

    val_90_m = float(in90["valor_cop"].sum() / 1e6)
    val_12_m = float(in12["valor_cop"].sum() / 1e6)
    prev_q_avg_m = float(prev_q["valor_cop"].sum() / 4.0 / 1e6)
    estacionalidad = (val_90_m / prev_q_avg_m) if prev_q_avg_m > 0 else (1.2 if val_90_m > 0 else 0.5)
    estacionalidad = float(np.clip(estacionalidad, 0.3, 2.5))

    top_prod = (
        mine.groupby("producto")["valor_cop"].sum().idxmax()
        if len(mine)
        else "Implementación documental"
    )
    # Demanda objetivo trimestral en millones COP (incluye ceros en inactivos -> ideal para Tweedie)
    demanda_valor = (0.65 * val_90_m + 0.35 * (val_12_m / 4.0)) if len(mine) else 0.0

    rows_cust.append({
        "id": c["id"],
        "empresa": c["empresa"],
        "ciudad": c["ciudad"],
        "sector": c["sector"],
        "canal": c["canal"],
        "frecuencia_compra": freq_12m,
        "ticket_promedio": ticket_m,
        "recencia_dias": recencia,
        "variedad_categorias": variedad,
        "indice_estacionalidad": estacionalidad,
        "producto_recomendado": top_prod,
        "demanda_valor": demanda_valor,
    })

df = pd.DataFrame(rows_cust)
features = [
    "frecuencia_compra",
    "ticket_promedio",
    "recencia_dias",
    "variedad_categorias",
    "indice_estacionalidad",
]

# Matriz de productos Green Services (10 líneas de servicio)
rows_prod = []
for prod, grp in df_conf.groupby("producto"):
    days_ago = (REF_DT - grp["fecha_dt"]).dt.days
    val_90_m = float(grp[days_ago < 91]["valor_cop"].sum() / 1e6)
    val_prev_m = float(grp[(days_ago >= 91) & (days_ago < 456)]["valor_cop"].sum() / 4.0 / 1e6)
    tendencia = ((val_90_m - val_prev_m) / val_prev_m) if val_prev_m > 0 else 0.0
    demanda_prod_m = 0.6 * val_90_m + 0.4 * val_prev_m
    rows_prod.append({
        "producto": prod,
        "ventas_confirmadas": int(len(grp)),
        "unidades": int(grp["cantidad"].sum()),
        "clientes_unicos": int(grp["customerId"].nunique()),
        "valor_historico_cop": float(grp["valor_cop"].sum()),
        "ticket_promedio_m": float(grp["valor_cop"].mean() / 1e6),
        "tendencia_90d": float(tendencia),
        "demanda_valor": float(demanda_prod_m),
    })
df_prod = pd.DataFrame(rows_prod)
print(df[features + ["demanda_valor"]].describe().round(2))

In [ ]:
# 4. PCA + K-Means + GLM Tweedie + PyTorch Mixture of Experts (MoE)
torch.manual_seed(42)
np.random.seed(42)

# --- 4.1 PCA y K-Means sobre clientes ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df[["pca_1", "pca_2", "pca_3"]] = X_pca

n_clusters = 3
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init="auto")
raw_clusters = kmeans.fit_predict(X_pca)

# Reordenar IDs de clúster por demanda promedio descendente (0=VIP, 1=Recurrente, 2=Riesgo)
order = (
    pd.Series(df["demanda_valor"]).groupby(raw_clusters).mean().sort_values(ascending=False).index.tolist()
)
remap = {old: new for new, old in enumerate(order)}
df["cluster"] = [remap[c] for c in raw_clusters]
sil_score = silhouette_score(X_pca, df["cluster"])

CLUSTER_LABELS = {
    0: "Alto Valor / VIP",
    1: "Recurrentes / Crecimiento",
    2: "En Riesgo / Reactivación",
}
EXPERT_NAMES = [
    "Experto 0 · Alto Valor",
    "Experto 1 · Recurrente",
    "Experto 2 · Reactivación",
]
df["cluster_label"] = df["cluster"].map(CLUSTER_LABELS)

print(f"Varianza explicada por PCA: {pca.explained_variance_ratio_.sum():.2%} | Silhouette: {sil_score:.3f}")
print("Distribución de clusters:\n", df["cluster_label"].value_counts(), "\n")

# --- 4.2 GLM Tweedie (var_power=1.5, enlace log) por clúster ---
glm_models = {}
df["pred_glm"] = 0.0
for c in range(n_clusters):
    subset = df[df["cluster"] == c]
    formula = "demanda_valor ~ pca_1 + pca_2 + pca_3 + indice_estacionalidad"
    try:
        model = smf.glm(
            formula=formula,
            data=subset,
            family=sm.families.Tweedie(var_power=1.5, link=sm.families.links.Log()),
        ).fit()
        preds = np.clip(model.predict(subset), 0.0, None)
    except Exception:
        # Respaldo regularizado si un clúster tuviera varianza degenerada
        model = smf.glm(
            formula="demanda_valor ~ pca_1 + pca_2",
            data=subset,
            family=sm.families.Tweedie(var_power=1.5, link=sm.families.links.Log()),
        ).fit_regularized(alpha=0.05)
        preds = np.clip(model.predict(subset), 0.0, None)
    glm_models[c] = model
    df.loc[df["cluster"] == c, "pred_glm"] = preds

# --- 4.3 Mixture of Experts (MoE) en PyTorch (normalizado para evitar colapso de pesos) ---
class ExpertNetwork(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x)


class GreenSightMoE(nn.Module):
    def __init__(self, input_dim: int, num_experts: int = 3):
        super().__init__()
        self.experts = nn.ModuleList([ExpertNetwork(input_dim) for _ in range(num_experts)])
        self.gate = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, num_experts),
            nn.Softmax(dim=-1),
        )

    def forward(self, x):
        gate_weights = self.gate(x)
        expert_outputs = torch.cat([expert(x) for expert in self.experts], dim=-1)
        output = torch.sum(gate_weights * expert_outputs, dim=-1, keepdim=True)
        return output, gate_weights


moe_features = ["pca_1", "pca_2", "pca_3", "pred_glm"]
moe_scaler = StandardScaler()
X_moe_np = moe_scaler.fit_transform(df[moe_features])
y_mean, y_std = float(df["demanda_valor"].mean()), float(df["demanda_valor"].std() or 1.0)
y_moe_np = (df["demanda_valor"].values - y_mean) / y_std

X_moe = torch.tensor(X_moe_np, dtype=torch.float32)
y_moe = torch.tensor(y_moe_np, dtype=torch.float32).unsqueeze(1)
cluster_onehot = torch.nn.functional.one_hot(torch.tensor(df["cluster"].values), num_classes=n_clusters).float()

moe_model = GreenSightMoE(input_dim=len(moe_features), num_experts=n_clusters)
optimizer = optim.Adam(moe_model.parameters(), lr=0.02)
criterion = nn.MSELoss()

for epoch in range(350):
    optimizer.zero_grad()
    preds, gate_w = moe_model(X_moe)
    loss_mse = criterion(preds, y_moe)
    # Regularización de especialización suave por clúster para evitar colapso [1,0,0]
    loss_gate = criterion(gate_w, cluster_onehot)
    loss = loss_mse + 0.35 * loss_gate
    loss.backward()
    optimizer.step()

with torch.no_grad():
    pred_norm, weights_moe = moe_model(X_moe)
    df["pred_moe"] = np.clip(pred_norm.squeeze().numpy() * y_std + y_mean, 0.0, None)
    weights_np = weights_moe.numpy()
    for e in range(n_clusters):
        df[f"peso_exp_{e}"] = weights_np[:, e]

mse_val = float(mean_squared_error(df["demanda_valor"], df["pred_moe"]))
rmse_val = float(np.sqrt(mse_val))
mae_val = float(mean_absolute_error(df["demanda_valor"], df["pred_moe"]))
r2_val = float(r2_score(df["demanda_valor"], df["pred_moe"]))
print(f"MoE completado | MSE: {mse_val:.2f} | RMSE: {rmse_val:.2f} | MAE: {mae_val:.2f} | R2: {r2_val:.4f}")

# --- 4.4 Pipeline equivalente sobre los 10 productos Green Services ---
prod_features = ["ventas_confirmadas", "unidades", "clientes_unicos", "ticket_promedio_m", "tendencia_90d"]
p_scaler = StandardScaler()
Xp_scaled = p_scaler.fit_transform(df_prod[prod_features])
Xp_pca = PCA(n_components=3, random_state=42).fit_transform(Xp_scaled)
df_prod[["pca_1", "pca_2", "pca_3"]] = Xp_pca
p_raw_cl = KMeans(n_clusters=3, random_state=42, n_init="auto").fit_predict(Xp_pca)
p_order = pd.Series(df_prod["demanda_valor"]).groupby(p_raw_cl).mean().sort_values(ascending=False).index.tolist()
df_prod["cluster_demanda"] = [ {old: new for new, old in enumerate(p_order)}[c] for c in p_raw_cl ]

p_glm = smf.glm(
    formula="demanda_valor ~ pca_1 + pca_2 + pca_3 + tendencia_90d",
    data=df_prod,
    family=sm.families.Tweedie(var_power=1.5, link=sm.families.links.Log()),
).fit()
df_prod["pred_glm"] = np.clip(p_glm.predict(df_prod), 0.0, None)

p_moe_scaler = StandardScaler()
Xp_moe = torch.tensor(p_moe_scaler.fit_transform(df_prod[["pca_1", "pca_2", "pca_3", "pred_glm"]]), dtype=torch.float32)
py_mean, py_std = float(df_prod["demanda_valor"].mean()), float(df_prod["demanda_valor"].std() or 1.0)
yp_moe = torch.tensor((df_prod["demanda_valor"].values - py_mean) / py_std, dtype=torch.float32).unsqueeze(1)
p_onehot = torch.nn.functional.one_hot(torch.tensor(df_prod["cluster_demanda"].values), num_classes=3).float()

prod_moe_model = GreenSightMoE(input_dim=4, num_experts=3)
p_opt = optim.Adam(prod_moe_model.parameters(), lr=0.02)
for _ in range(350):
    p_opt.zero_grad()
    pp, pw = prod_moe_model(Xp_moe)
    (criterion(pp, yp_moe) + 0.35 * criterion(pw, p_onehot)).backward()
    p_opt.step()

with torch.no_grad():
    pp_norm, pw_moe = prod_moe_model(Xp_moe)
    df_prod["pred_moe"] = np.clip(pp_norm.squeeze().numpy() * py_std + py_mean, 0.0, None)
    pw_np = pw_moe.numpy()
    for e in range(3):
        df_prod[f"peso_exp_{e}"] = pw_np[:, e]

df_prod = df_prod.sort_values("pred_moe", ascending=False).reset_index(drop=True)
print("\nTop productos por demanda optimizada MoE (M COP):")
print(df_prod[["producto", "pred_glm", "pred_moe", "tendencia_90d"]].round(2))

In [ ]:
# 5. Gráficas en Colab (sincronizadas 1:1 con el Dashboard Green 360°)
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
palette = {0: "#0F5132", 1: "#16A34A", 2: "#0E7490"}

# 1) Proyección PCA 2D por Clúster K-Means
for cid, label in CLUSTER_LABELS.items():
    sub = df[df["cluster"] == cid]
    axes[0, 0].scatter(sub["pca_1"], sub["pca_2"], label=f"{label} ({len(sub)})", color=palette[cid], alpha=0.85, s=60)
axes[0, 0].set_title(f"PCA 2D + K-Means (Varianza acumulada: {pca.explained_variance_ratio_.sum():.1%})")
axes[0, 0].set_xlabel("PC1 (Frecuencia / Recencia)")
axes[0, 0].set_ylabel("PC2 (Estacionalidad / Variedad)")
axes[0, 0].legend()

# 2) Demanda Total por Clúster: GLM Tweedie vs MoE
cl_summary = df.groupby("cluster")[["pred_glm", "pred_moe"]].sum()
x_idx = np.arange(len(cl_summary))
axes[0, 1].bar(x_idx - 0.18, cl_summary["pred_glm"], width=0.35, label="GLM Tweedie", color="#0E7490")
axes[0, 1].bar(x_idx + 0.18, cl_summary["pred_moe"], width=0.35, label="MoE Optimizado", color="#16A34A")
axes[0, 1].set_xticks(x_idx)
axes[0, 1].set_xticklabels([CLUSTER_LABELS[i] for i in cl_summary.index], rotation=12)
axes[0, 1].set_title("Demanda Estimada por Clúster (M COP)")
axes[0, 1].legend()

# 3) Demanda por Producto Green Services: GLM vs MoE
top_p = df_prod.head(8).iloc[::-1]
y_idx = np.arange(len(top_p))
axes[1, 0].barh(y_idx - 0.18, top_p["pred_glm"], height=0.35, label="GLM Tweedie", color="#0E7490")
axes[1, 0].barh(y_idx + 0.18, top_p["pred_moe"], height=0.35, label="MoE Optimizado", color="#0F5132")
axes[1, 0].set_yticks(y_idx)
axes[1, 0].set_yticklabels(top_p["producto"])
axes[1, 0].set_title("Productos de Mayor Necesidad (M COP)")
axes[1, 0].legend()

# 4) Heatmap de Pesos de Expertos MoE por Clúster
heat = df.groupby("cluster")[["peso_exp_0", "peso_exp_1", "peso_exp_2"]].mean()
heat.index = [CLUSTER_LABELS[i] for i in heat.index]
heat.columns = EXPERT_NAMES
sns.heatmap(heat, annot=True, fmt=".2f", cmap="YlGn", ax=axes[1, 1], vmin=0, vmax=1)
axes[1, 1].set_title("Pesos Promedio de Enrutamiento MoE")

plt.tight_layout()
plt.show()

In [ ]:
# 6. Inferencia Unificada + Exportación JSON para Dashboard + Hugging Face + n8n
def predecir_greensight(cliente_dict: dict, n8n_webhook_url: str = None) -> dict:
    """Ejecuta Scaler -> PCA -> K-Means -> GLM Tweedie -> MoE y dispara n8n si aplica."""
    input_df = pd.DataFrame([cliente_dict])
    x_sc = scaler.transform(input_df[features])
    x_p = pca.transform(x_sc)

    raw_cid = int(kmeans.predict(x_p)[0])
    cluster_id = remap[raw_cid]

    glm_input = pd.DataFrame({
        "pca_1": [x_p[0, 0]],
        "pca_2": [x_p[0, 1]],
        "pca_3": [x_p[0, 2]],
        "indice_estacionalidad": [cliente_dict["indice_estacionalidad"]],
    })
    pred_glm_val = float(max(0.0, glm_models[cluster_id].predict(glm_input)[0]))

    moe_raw = np.array([[x_p[0, 0], x_p[0, 1], x_p[0, 2], pred_glm_val]])
    moe_in = torch.tensor(moe_scaler.transform(moe_raw), dtype=torch.float32)
    with torch.no_grad():
        pred_n, pesos_exp = moe_model(moe_in)
        pred_final = float(max(0.0, pred_n.item() * y_std + y_mean))

    resultado = {
        "segmento_cluster": cluster_id,
        "cluster_label": CLUSTER_LABELS[cluster_id],
        "demanda_estimada_glm_m_cop": round(pred_glm_val, 2),
        "demanda_optimizada_moe_m_cop": round(pred_final, 2),
        "pesos_expertos_moe": [round(float(w), 4) for w in pesos_exp.squeeze().tolist()],
        "recomendacion_producto": (
            "Alta prioridad / Reposición inmediata"
            if (cluster_id == 0 or pred_final >= 15.0)
            else ("Reactivación prioritaria / Contacto comercial" if cluster_id == 2 else "Demanda estándar")
        ),
    }

    if n8n_webhook_url:
        try:
            requests.post(n8n_webhook_url, json=resultado, timeout=4)
            resultado["n8n_status"] = "Enviado a webhook n8n"
        except Exception as exc:
            resultado["n8n_status"] = f"Webhook no alcanzado: {exc}"

    return resultado

# Prueba unitaria corregida (en la versión anterior decía ejemplo_cliente)
ejemplo_cliente = {
    "frecuencia_compra": 12.0,
    "ticket_promedio": 5.5,
    "recencia_dias": 14.0,
    "variedad_categorias": 5.0,
    "indice_estacionalidad": 1.25,
}
print("Resultado de inferencia:", json.dumps(predecir_greensight(ejemplo_cliente), indent=2, ensure_ascii=False))

# Exportar artefacto JSON compatible 1:1 con el Dashboard Green 360°
med_prod_moe = float(df_prod["pred_moe"].median())
export_payload = {
    "metadata": {
        "modelName": "GreenSight ML · PCA + K-Means + GLM Tweedie + MoE",
        "source": "colab/Machine_Learning_GreenSight.ipynb",
        "generatedAt": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "fechaReferencia": FECHA_REFERENCIA,
        "nClientes": int(len(df)),
        "nVentasConfirmadas": int(len(df_conf)),
        "nProductos": int(len(df_prod)),
        "nClusters": n_clusters,
        "tweedieVarPower": 1.5,
        "colabNotebookUrl": "https://colab.research.google.com/drive/1Ulr5twAm2ynI8B-U_zxaAy-HGUpNUu5Q",
    },
    "pca": {
        "explainedVarianceRatio": [round(float(x), 4) for x in pca.explained_variance_ratio_],
        "cumulativeVariance": round(float(pca.explained_variance_ratio_.sum()), 4),
        "features": features,
        "loadings": {
            f"pca_{k+1}": [round(float(v), 4) for v in pca.components_[k]]
            for k in range(3)
        },
    },
    "moeMetrics": {
        "mse": round(mse_val, 2),
        "rmse": round(rmse_val, 2),
        "mae": round(mae_val, 2),
        "r2": round(r2_val, 4),
        "silhouetteScore": round(float(sil_score), 4),
        "expertNames": EXPERT_NAMES,
    },
    "clusters": [
        {
            "clusterId": int(cid),
            "label": CLUSTER_LABELS[cid],
            "count": int((df["cluster"] == cid).sum()),
            "pct": round(float((df["cluster"] == cid).mean()), 4),
            "avgRecenciaDias": round(float(df[df["cluster"] == cid]["recencia_dias"].mean()), 1),
            "avgFrecuencia12m": round(float(df[df["cluster"] == cid]["frecuencia_compra"].mean()), 1),
            "avgTicketCOP": int(round(float(df[df["cluster"] == cid]["ticket_promedio"].mean() * 1e6))),
            "demandaGlmTotalCOP": int(round(float(df[df["cluster"] == cid]["pred_glm"].sum() * 1e6))),
            "demandaMoeTotalCOP": int(round(float(df[df["cluster"] == cid]["pred_moe"].sum() * 1e6))),
            "pesosExpertosPromedio": [
                round(float(df[df["cluster"] == cid][f"peso_exp_{e}"].mean()), 4) for e in range(3)
            ],
            "centroidPca": [round(float(v), 4) for v in df[df["cluster"] == cid][["pca_1", "pca_2", "pca_3"]].mean()],
            "recomendacion": (
                "Expansión comercial y upselling de suite documental" if cid == 0
                else ("Fidelización y venta cruzada de servicios complementarios" if cid == 1
                      else "Campaña prioritaria de reactivación y soporte preventivo")
            ),
            "prioridad": "media" if cid == 1 else "alta",
        }
        for cid in range(n_clusters)
    ],
    "customers": [
        {
            "id": r["id"],
            "empresa": r["empresa"],
            "ciudad": r["ciudad"],
            "sector": r["sector"],
            "canal": r["canal"],
            "clusterId": int(r["cluster"]),
            "clusterLabel": r["cluster_label"],
            "pca": [round(float(r["pca_1"]), 4), round(float(r["pca_2"]), 4), round(float(r["pca_3"]), 4)],
            "frecuencia12m": int(r["frecuencia_compra"]),
            "recenciaDias": int(r["recencia_dias"]),
            "ticketPromedioCOP": int(round(float(r["ticket_promedio"] * 1e6))),
            "demandaGlmCOP": int(round(float(r["pred_glm"] * 1e6))),
            "demandaMoeCOP": int(round(float(r["pred_moe"] * 1e6))),
            "pesosExpertos": [round(float(r[f"peso_exp_{e}"]), 4) for e in range(3)],
            "productoRecomendado": r["producto_recomendado"],
            "prioridad": "media" if (r["cluster"] == 1 and r["pred_moe"] < 15.0) else "alta",
            "recomendacion": (
                "Alta prioridad / Reposición inmediata"
                if (r["cluster"] == 0 or r["pred_moe"] >= 15.0)
                else ("Reactivación prioritaria / Contacto comercial" if r["cluster"] == 2 else "Demanda estándar / Seguimiento periódico")
            ),
        }
        for _, r in df.iterrows()
    ],
    "products": [
        {
            "producto": r["producto"],
            "ventasConfirmadas": int(r["ventas_confirmadas"]),
            "unidades": int(r["unidades"]),
            "clientesUnicos": int(r["clientes_unicos"]),
            "valorHistoricoCOP": int(round(float(r["valor_historico_cop"]))),
            "ticketPromedioCOP": int(round(float(r["ticket_promedio_m"] * 1e6))),
            "tendencia90d": round(float(r["tendencia_90d"]), 4),
            "clusterDemanda": int(r["cluster_demanda"]),
            "demandaGlmCOP": int(round(float(r["pred_glm"] * 1e6))),
            "demandaMoeCOP": int(round(float(r["pred_moe"] * 1e6))),
            "pesosExpertos": [round(float(r[f"peso_exp_{e}"]), 4) for e in range(3)],
            "prioridad": "alta" if (r["pred_moe"] >= med_prod_moe or r["tendencia_90d"] >= 0.08) else "media",
            "recomendacion": "Alta prioridad / Reposición inmediata" if (r["pred_moe"] >= med_prod_moe or r["tendencia_90d"] >= 0.08) else "Demanda estándar",
        }
        for _, r in df_prod.iterrows()
    ],
}

Path("greensight_insights.json").write_text(json.dumps(export_payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("Exportado greensight_insights.json listo para Green 360°.")
try:
    from google.colab import files
    # Descomentar para descargar automáticamente al ejecutar en Colab:
    # files.download("greensight_insights.json")
except ImportError:
    pass

In [ ]:
# 7. Servidor MCP + Endpoint FastAPI opcional para conectar Colab en vivo al Dashboard
# Guarda y ejecuta como servicio o expón con cloudflared/ngrok desde Colab si deseas conexión en tiempo real.

MCP_SERVER_CODE = '''
from mcp.server.fastmcp import FastMCP
import json
from pathlib import Path

mcp = FastMCP("GreenSight-ML-Predictor")

@mcp.tool()
def obtener_resumen_greensight() -> dict:
    """Retorna las métricas PCA, clústeres K-Means y predicciones MoE de Green Services."""
    data = json.loads(Path("greensight_insights.json").read_text(encoding="utf-8"))
    return {
        "metadata": data["metadata"],
        "pca": data["pca"],
        "moeMetrics": data["moeMetrics"],
        "clusters": data["clusters"],
        "top_products": data["products"][:5],
    }

@mcp.tool()
def generar_prompt_moe_campana(cluster_id: int, objetivo_operativo: str) -> dict:
    """Construye un prompt especializado por Mixture of Experts (MoE) solo para Green Services."""
    data = json.loads(Path("greensight_insights.json").read_text(encoding="utf-8"))
    cl = next((c for c in data["clusters"] if c["clusterId"] == cluster_id), data["clusters"][0])
    top_prod = data["products"][0]["producto"]
    return {
        "cluster": cl["label"],
        "pesos_expertos": cl["pesosExpertosPromedio"],
        "producto_foco": top_prod,
        "prompt_moe": (
            f"Actúa como asesor comercial senior de Green Services. "
            f"Dirígete al segmento {cl['label']} (pesos MoE {cl['pesosExpertosPromedio']}), "
            f"enfocándote en {top_prod}. Objetivo operativo: {objetivo_operativo}."
        ),
    }

if __name__ == "__main__":
    mcp.run()
'''

print("Código de Servidor MCP generado. En Docker local usa backend/mcp_greensight_server.py y backend/app/main.py.")